# Util embedding experiments
Flow: load stimuli -> get embeddings -> fit a direction per subset -> compare directions.

In [1]:
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import datasets
import embed

/home/matthew/Projects/2025_moral_feature_modeling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load stimuli
`SUBSETS` maps each subset label -> {"texts": list[str], "y": float array} for the directions we fit.

In [2]:
SUBSETS = {}  # label -> {"texts": list[str], "y": np.ndarray(float)}
# Small datasets first (fast feedback); large ETHICS train splits last.

# franken
exp1, exp2 = datasets.load_franken()
SUBSETS["franken-valence"] = {"texts": exp1["target"].tolist(), "y": exp1["avg_likert_rating"].to_numpy(float)}
SUBSETS["franken-good_vs_harm"] = {"texts": exp1["target"].tolist(), "y": (exp1["type"] == "good").to_numpy(float)}
SUBSETS["franken-severe_vs_mild"] = {"texts": exp1["target"].tolist(), "y": (exp1["strength"] == "severe").to_numpy(float)}
SUBSETS["franken-permissibility"] = {"texts": exp2["text"].tolist(), "y": exp2["avg_permissibility_rating"].to_numpy(float)}
SUBSETS["franken-intention"] = {"texts": exp2["text"].tolist(), "y": exp2["avg_intention_rating"].to_numpy(float)}

# nie / MoCa
nie = datasets.load_nie()
SUBSETS["nie-acceptability"] = {"texts": nie["text"].tolist(), "y": nie["p_yes"].to_numpy(float)}
for col, pos in [("causal_role", "Means"), ("personal_force", "Personal"),
                 ("evitability", "Inevitable"), ("beneficiary", "Other-beneficial")]:
    sub = nie.dropna(subset=[col])
    if sub[col].nunique() > 1:
        SUBSETS[f"nie-{col}"] = {"texts": sub["text"].tolist(), "y": (sub[col] == pos).to_numpy(float)}

# AITA
aita = datasets.load_aita()
SUBSETS["AITA-utility"] = {"texts": aita["outcome"].tolist(), "y": aita["utility"].to_numpy(float)}

# Holmes-Rahe
hr = datasets.load_holmes_rahe()
SUBSETS["Holmes-Rahe"] = {"texts": hr["event"].tolist(), "y": hr["lcu"].to_numpy(float)}

# GBD
gbd = datasets.load_gbd()
SUBSETS["GBD"] = {"texts": gbd["lay_description"].tolist(), "y": gbd["weight"].to_numpy(float)}

# ETHICS (large train splits - slowest to embed)
ethics = datasets.load_ethics()
a, b = ethics["utilitarianism"]["train"]["more_pleasant"], ethics["utilitarianism"]["train"]["less_pleasant"]
SUBSETS["ETHICS-util"] = {"texts": list(a) + list(b), "y": np.r_[np.ones(len(a)), np.zeros(len(b))]}

cm = ethics["commonsense"]["train"]
cm_short = cm[cm["is_short"]]
SUBSETS["ETHICS-cm"] = {"texts": cm_short["input"].tolist(), "y": cm_short["label"].to_numpy(float)}

deon = ethics["deontology"]["train"]
deon_text = (deon["scenario"] + " " + deon["excuse"]).tolist()
SUBSETS["ETHICS-deon"] = {"texts": deon_text, "y": deon["label"].to_numpy(float)}

for label, sub in SUBSETS.items():
    print(f"{label:24s} n={len(sub['texts']):5d}")

franken-valence          n=   80
franken-good_vs_harm     n=   80
franken-severe_vs_mild   n=   80
franken-permissibility   n=   80
franken-intention        n=   80
nie-acceptability        n=   44
nie-causal_role          n=   35
nie-personal_force       n=   35
nie-evitability          n=   35
nie-beneficiary          n=   35
AITA-utility             n=   59
Holmes-Rahe              n=   43
GBD                      n=  203
ETHICS-util              n=27476
ETHICS-cm                n= 6661
ETHICS-deon              n=18164


## 2. Get embeddings
Switch `MODEL` to any name in `embed.OPENAI_MODELS` or `embed.QWEN_MODELS` to change model.

In [3]:
import os
MODEL = "Qwen/Qwen3-Embedding-4B"  # or any of embed.OPENAI_MODELS / embed.QWEN_MODELS
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # only needed if MODEL is an OpenAI model

embeddings = {}
for label, sub in SUBSETS.items():
    emb = embed.embed_dataset(sub["texts"], dataset=label, model=MODEL, api_key=OPENAI_API_KEY)
    X = np.array([emb[str(t)] for t in sub["texts"]])
    embeddings[label] = X
    print(f"{label:24s} {X.shape}")

franken-valence          (80, 2560)
franken-good_vs_harm     (80, 2560)
franken-severe_vs_mild   (80, 2560)
franken-permissibility   (80, 2560)
franken-intention        (80, 2560)
nie-acceptability        (44, 2560)
nie-causal_role          (35, 2560)
nie-personal_force       (35, 2560)
nie-evitability          (35, 2560)
nie-beneficiary          (35, 2560)
AITA-utility             (59, 2560)
Holmes-Rahe              (43, 2560)
GBD                      (203, 2560)


ETHICS-util              (27476, 2560)
ETHICS-cm                (6661, 2560)


ETHICS-deon              (18164, 2560)


## 3. Fit directions
Two simple methods to compare: ridge regression of the embeddings on `y`, and a plain
mean-difference between the high and low halves (split at the median for continuous `y`).

In [4]:
def ridge_dir(X, y, alpha=50.0):
    Xc = X - X.mean(0)
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ (y - y.mean()))
    return w / (np.linalg.norm(w) + 1e-9)

def meandiff_dir(X, y):
    classes = np.unique(y)
    if len(classes) == 2:            # binary: split by class membership, not by a threshold
        hi, lo = X[y == classes[1]].mean(0), X[y == classes[0]].mean(0)
    else:                            # continuous: split at the median
        thresh = np.median(y)
        hi, lo = X[y >= thresh].mean(0), X[y < thresh].mean(0)
    d = hi - lo
    return d / (np.linalg.norm(d) + 1e-9)

METHOD = "ridge"  # or "meandiff"

directions = {}
for label, sub in SUBSETS.items():
    X = embeddings[label]
    d = ridge_dir(X, sub["y"]) if METHOD == "ridge" else meandiff_dir(X, sub["y"])
    directions[label] = d

## 4. Correlation between y and projection
For each subset, project its own embeddings onto its fitted direction and correlate
with the true `y`. In-sample (not held-out) - a quick sanity check that the direction
actually points the way it should, not a decodability estimate.

In [5]:
labels_c, rs = [], []
for label, sub in SUBSETS.items():
    proj = embeddings[label] @ directions[label]
    rs.append(np.corrcoef(proj, sub["y"])[0, 1])
    labels_c.append(label)

fig = go.Figure(go.Bar(x=labels_c, y=rs, text=rs, texttemplate="%{text:.2f}", textposition="outside"))
fig.update_layout(title="In-sample correlation between y and projection onto fitted direction",
                   yaxis_title="Pearson r", xaxis_tickangle=-60, width=900, height=500)
fig.show()

## 5. Cosine similarity between directions

In [6]:
labels = list(directions.keys())
D = np.array([directions[l] for l in labels])
C = D @ D.T

fig = px.imshow(C, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Direction x direction cosine similarity", width=900, height=850)
fig.show()

## 6. Correlations between labels and directions (Cross-Dataset)
Like section 4, but cross-dataset: for every (labels_i, direction_j) pair, project
subset i's embeddings onto subset j's fitted direction and correlate with subset i's
own `y`. Row = whose data/labels, column = whose direction. The diagonal reproduces
section 4's numbers; off-diagonal cells show how well one dataset's direction
generalizes to another's labels (e.g. franken-valence data projected onto the
nie-causal_role direction, correlated with the franken-valence labels).

In [7]:
Cross = np.zeros((len(labels), len(labels)))
for i, li in enumerate(labels):
    Xi, yi = embeddings[li], SUBSETS[li]["y"]
    for j, lj in enumerate(labels):
        proj = Xi @ directions[lj]
        Cross[i, j] = np.corrcoef(proj, yi)[0, 1]

fig = px.imshow(Cross, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Correlation between y and projection, cross-dataset (row = data, column = direction)",
                   width=900, height=850)
fig.update_xaxes(title="direction")
fig.update_yaxes(title="data / labels")
fig.show()